# Research source snapshot
Outputs and execution metadata have been removed. Read the repository README before running. These exploratory notebooks are not validated end to end. Only load trusted local pickle files.


In [ ]:
import os
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("RESEARCH_DATA_DIR", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("RESEARCH_OUTPUT_DIR", str(REPOSITORY_ROOT / "outputs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
indian_villages=pd.read_csv(str(DATA_ROOT / "India.csv"),low_memory=False)

In [ ]:
indian_villages.head()

In [ ]:
indian_villages

In [ ]:
indian_villages.columns

In [ ]:
indian_villages.info()

In [ ]:
indian_villages['10'].value_counts()

In [ ]:
rural_villages = indian_villages[indian_villages['10'] == 'Rural']

In [ ]:
rural_villages.info()

In [ ]:
rural_villages

In [ ]:
urban_villages = indian_villages[indian_villages['10'] == 'Urban']

In [ ]:
urban_villages

In [ ]:
gt_2011=pd.read_csv(str(DATA_ROOT / "gt_2011.csv"),low_memory=False)

In [ ]:
gt_2011.head()

In [ ]:
gt_2011.columns

In [ ]:
filtered_rural_indian_villages = rural_villages[rural_villages['7'].isin(gt_2011['Unnamed: 0'])]

In [ ]:
filtered_rural_indian_villages.head()

In [ ]:
filtered_rural_indian_villages.info()

**EXTRACTING COLUMNS FOR RWI**

In [ ]:
import pandas as pd
column_categories = {
# Assuming 'rural_villages' is your DataFrame containing the data
  # Roof Material
   'Roof_Material_rud': ['23','24'],
   'Roof_Material_int': [ '25', '26', '27'],
   'Roof_Material_adv': ['28', '29', '30'],
   # Wall Material
   'Wall_Material_rud': ['32', '33', '34'],
   'Wall_Material_int': ['35', '36', '37'],
   'Wall_Material_adv': ['38', '39', '40'],
   # Floor Material
   'Floor_Material_rud': ['42', '43'],
   'Floor_Material_int': ['44', '45'],
   'Floor_Material_adv': ['46', '47'],
   # Number of Dwelling Rooms (NEW)
   'Num_Dwelling_Rooms_rud': ['49'],
   'Num_Dwelling_Rooms_int': ['50', '51'],
   'Num_Dwelling_Rooms_adv': ['52', '53', '54', '55'],
   # Ownership Status
   'Ownership_Status_rud': ['65'],
   'Ownership_Status_int': ['64'],
   'Ownership_Status_adv': ['63'],
   # Drinking Water
   'Drinking_Water_rud': ['78', '79','80'],
   'Drinking_Water_int': ['74','75','76', '77'],
   'Drinking_Water_adv': ['72', '73'],
   # Location of Drinking Source (NEW)
   'Location_Drinking_Source_rud': ['84'],
   'Location_Drinking_Source_int': ['83'],
   'Location_Drinking_Source_adv': ['82'],
   # Lighting
   'Lighting_rud': ['90'],
   'Lighting_int': ['86'],
   'Lighting_adv': ['85'],
   # Latrine
   'Latrine_rud': ['100'],
   'Latrine_int': ['96', '95'],
   'Latrine_adv': ['93', '92'],
   # Bathing Facility (NEW)
   'Bathing_Facility_rud': ['105'],
   'Bathing_Facility_int': ['104'],
   'Bathing_Facility_adv': ['103'],
   # Waste Water outlet connected to (NEW)
   'Waste_Water_Outlet_rud': ['108'],
   'Waste_Water_Outlet_int': ['107'],
   'Waste_Water_Outlet_adv': ['106'],
   # Cooking Fuel
   'Cooking_Fuel_rud': ['109', '110', '118'],
   'Cooking_Fuel_int': ['111','112','113'],
   'Cooking_Fuel_adv': ['114', '115', '116'],
   # Kitchen facilities
   'Kitchen_Facilities_rud': ['125', '126'],
   'Kitchen_Facilities_int': ['122', '124'],
   'Kitchen_Facilities_adv': ['121']
}

data_subset_rwi_new = {}  # Initialize as a dictionary, not a list

# Iterate through column categories, summing values for each category
for category, cols in column_categories.items():
    data_subset_rwi_new[category] = filtered_rural_indian_villages[cols].sum(axis=1)

columns_to_add_independently = {
  'Banking': ['127'],
  'Radio':['128'],
  'Television':['129'],
  'Computer':['130','131'],
  'landline_only':['132'],
  'mobile_only':['133'],
  'landline_and_mobile':['134'],
  '2w':['135','136'],
  '4w':['137'],
  'all_assets':['138'],
  'Noasset':['139']
}

# Iterate through the columns and add them to the DataFrame
for col, col_indices in columns_to_add_independently.items():
    data_subset_rwi_new[col] = filtered_rural_indian_villages[col_indices].sum(axis=1)

print(data_subset_rwi_new)

In [ ]:
# After creating and populating the data_subset_rwi_new dictionary
column_names = data_subset_rwi_new.keys()
print(column_names)

In [ ]:
import pandas as pd

# Convert the dictionary to a DataFrame
data_subset_rwi_new_df = pd.DataFrame(data_subset_rwi_new)

# Now you can use DataFrame methods, including fillna
#data_subset_rwi_new_df.fillna(data_subset_rwi_new_df.mean(), inplace=True)

# Print the DataFrame's columns
print(data_subset_rwi_new_df.columns)

In [ ]:
data_subset_rwi_new_df.info()

**Standardize the Data**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_standardized = scaler.fit_transform(data_subset_rwi_new_df)

In [ ]:
print(data_standardized[:5])

In [ ]:
data_standardized

In [ ]:
import numpy as np

# Compute mean of each column
means = np.mean(data_standardized, axis=0)

# Compute standard deviation of each column
std_devs = np.std(data_standardized, axis=0)

print("Means:", means)
print("Standard Deviations:", std_devs)


**Principal Component Analysis (PCA)**

In [ ]:
from sklearn.decomposition import PCA

pca_first = PCA(n_components=1)
m=pca_first.fit_transform(data_standardized)

In [ ]:
import pandas as pd

# Convert the numpy array to a pandas DataFrame
df_pca = pd.DataFrame(m, columns=['Principal Component 1'])

# Use the describe method
print(df_pca.describe())

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Initialize PCA without specifying n_components
pca = PCA(n_components=10)

# Fit the standardized data
pca.fit(data_standardized)

# Get the explained variance ratio
explained_variance = pca.explained_variance_ratio_

# Plot the explained variance
plt.figure(figsize=(10, 5))
plt.bar(range(len(explained_variance)), explained_variance, alpha=0.5, align='center', label='Individual Explained Variance')
plt.ylabel('Explained Variance Ratio')
plt.xlabel('Principal Components')
plt.title('Explained Variance of Principal Components')
plt.legend(loc='best')
plt.tight_layout()
plt.show()


In [ ]:
print(data_standardized.mean())
print(data_standardized.std())

In [ ]:
import matplotlib.pyplot as plt

explained_variance = pca.explained_variance_ratio_

plt.figure(figsize=(10, 5))
# Plot all components in a light color
plt.bar(range(len(explained_variance)), explained_variance, alpha=0.3, align='center', label='Individual explained variance')
# Highlight the first component
plt.bar(0, explained_variance[0], alpha=0.7, align='center', color='red', label='First component explained variance')
plt.ylabel('Explained variance ratio')
plt.xlabel('Principal components')
plt.title('Explained Variance of Principal Components')
plt.legend(loc='best')
plt.tight_layout()
plt.show()


**Extract Factor Loadings**

In [ ]:
factor_loadings = pca_first.components_

In [ ]:
factor_loadings

In [ ]:
# Extracting the loadings for the first principal component
loadings_first_component = factor_loadings[0]

# Creating a DataFrame to map loadings to their respective columns
loadings_df = pd.DataFrame({
    'Column': data_subset_rwi_new_df.columns,
    'Loading': loadings_first_component
})

# Sorting the DataFrame by absolute loading values to see most influential variables
sorted_loadings = loadings_df.reindex(loadings_df.Loading.abs().sort_values(ascending=False).index)

print(sorted_loadings)

In [ ]:
import pandas as pd

# Extract positive and negative loadings
positive_loadings = loadings_df[loadings_df['Loading'] > 0].sort_values(by='Loading', ascending=False)
negative_loadings = loadings_df[loadings_df['Loading'] < 0].sort_values(by='Loading', ascending=True)

# Display positive and negative loadings
print("Positive Loadings:")
print(positive_loadings)

print("\nNegative Loadings:")
print(negative_loadings)

**Compute the RWI**

In [ ]:
rw_index = np.dot(data_standardized, factor_loadings.T)

In [ ]:
# Extracting the information columns
info_columns = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
data_info = filtered_rural_indian_villages[info_columns]

In [ ]:
data_info

In [ ]:
# Convert the rw_index (which is an ndarray) to a DataFrame for easy concatenation
rw_index_df = pd.DataFrame(rw_index, columns=["RWI"])

# Resetting the index for both DataFrames
data_info_reset = data_info.reset_index(drop=True)
rw_index_df_reset = rw_index_df.reset_index(drop=True)

# Concatenate the reset DataFrames
result_data = pd.concat([data_info_reset, rw_index_df_reset], axis=1)
result_data.index=data_info.index

In [ ]:
result_data

In [ ]:
result_data['wealth_rank'] = result_data['RWI'].rank(ascending=False)
result_data['wealth_quintile'] = pd.qcut(result_data['wealth_rank'], 5, labels=[1, 2, 3, 4, 5])

In [ ]:
result_data.head()

In [ ]:
result_data.columns

In [ ]:
result_data.info()

In [ ]:
mean_rwi=result_data['RWI'].mean()
std_rwi=result_data['RWI'].std()

In [ ]:
print(f"Mean of RWI: {mean_rwi}")
print(f"Standard Deviation of RWI: {std_rwi}")

In [ ]:
result_data.columns

In [ ]:
new_result_data=result_data.copy()

In [ ]:
# Get the name of the 7th column dynamically
index_column_name = new_result_data.columns[6]  # Indexing starts from 0, so the 7th column has index 6

# Set the 7th column as the index
new_result_data.set_index(index_column_name, inplace=True)

In [ ]:
new_result_data.head()

In [ ]:
import pickle
with open(str(OUTPUT_ROOT / "new_relative_wealth_index.pickle"), "wb") as f:
    pickle.dump(new_result_data, f)